In [1]:
# ============================================================
# SECTION 0 — Install & Imports
# ============================================================
!pip install ultralytics -q

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from ultralytics import YOLO

# ============================================================
# SECTION 1 — Config — FILL THESE IN
# ============================================================
DATASET = Path("/kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive")
IMAGES  = DATASET / "images"
SPLIT   = DATASET / "split_dataset"
CLASSES = DATASET / "classes_en.txt"

WORK = Path("/kaggle/working")
WORK.mkdir(exist_ok=True, parents=True)

# Where your 5 trained model outputs live (adjust to your actual dataset/model slugs)
MODEL_ROOT = Path("/kaggle/input/datasets/banutheghostbanana/results-hierchial-yolo26n")

MODEL_PATHS = {
    "Model B":       MODEL_ROOT / "model_B_outputs"       / "weights" / "best.pt",
    "Model A":       MODEL_ROOT / "model_A_outputs"       / "weights" / "best.pt",
    "A_v2_widegap":  MODEL_ROOT / "A_v2_widegap_outputs"  / "weights" / "best.pt",
    "A_v3_highcls":  MODEL_ROOT / "A_v3_highcls_outputs"  / "weights" / "best.pt",
    "A_v4_combined": MODEL_ROOT / "A_v4_combined_outputs" / "weights" / "best.pt",
}

for name, p in MODEL_PATHS.items():
    print(f"{name:<16}: {'FOUND' if p.exists() else 'MISSING'} -> {p}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.5 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model B         : FOUND -> /kaggle/input/datasets/banutheghostbanana/results-hierchial-yolo26n/model_B_outputs/weights/best.pt
Model A         : FOUND -> /kaggle/input/datasets/banutheghostbanana/results-hierchial-yolo26n/model_A_outputs/weights/best.pt
A_v2_widegap    : FOUND -> /kaggle/input/datasets/banutheghostbanana/results-hierchial-yolo26n/A_v2_widegap_outputs/weights/best.pt
A_v3_highcls    : FOUND -> /kaggle/input/datasets/banutheghostbanana/results-hierchial-yolo26n/A_v3_highcls_outputs/weights/best.pt
A_v4_

In [2]:
# ============================================================
# SECTION 2 — Data prep (path files + yaml)
# ============================================================
def write_paths(split_txt, out_txt):
    with open(split_txt, encoding="utf-8") as f:
        names = [l.strip() for l in f if l.strip()]
    paths = [str(IMAGES / name) for name in names]
    with open(out_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(paths))
    print(f"Written {len(paths)} paths -> {out_txt}")

write_paths(SPLIT / "train_files.txt", WORK / "train_paths.txt")
write_paths(SPLIT / "test_files.txt",  WORK / "val_paths.txt")

with open(CLASSES, encoding="utf-8") as f:
    class_names = [l.strip() for l in f if l.strip()]
n_classes = len(class_names)

import yaml
data_yaml = {
    "path":  str(WORK),
    "train": "train_paths.txt",
    "val":   "val_paths.txt",
    "nc":    n_classes,
    "names": class_names,
}
yaml_path = WORK / "data.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, allow_unicode=True, sort_keys=False)

print(f"Classes: {n_classes}, yaml -> {yaml_path}")

Written 2552 paths -> /kaggle/working/train_paths.txt
Written 639 paths -> /kaggle/working/val_paths.txt
Classes: 52, yaml -> /kaggle/working/data.yaml


In [3]:
# ============================================================
# SECTION 3 — Tier assignment (from EDA)
# ============================================================
counts = {
    0:312, 1:22, 2:470, 3:454, 4:33, 5:256, 6:182, 7:247,
    8:213, 9:43, 10:1071, 11:26, 12:280, 13:372, 14:85,
    15:29, 16:73, 17:105, 18:214, 19:29, 20:52, 21:11,
    22:131, 23:465, 24:56, 25:25, 26:80, 27:36, 28:27,
    29:45, 30:138, 31:38, 32:186, 33:19, 34:82, 35:87,
    36:75, 37:47, 38:189, 39:459, 40:210, 41:275, 42:21,
    43:27, 44:3, 45:195, 46:435, 47:16, 48:41, 49:221,
    50:121, 51:5
}
TIER_HIGH, TIER_MED = 100, 30
TIER = {}
for cid in range(n_classes):
    c = counts.get(cid, 0)
    if c >= TIER_HIGH: TIER[cid] = "Frequent"
    elif c >= TIER_MED: TIER[cid] = "Medium"
    else: TIER[cid] = "Rare"

def tier_mean(ap_array, tier_name):
    ids = [i for i in range(n_classes) if TIER[i] == tier_name]
    return np.mean([ap_array[i] for i in ids])

print(f"Tiers assigned for {len(TIER)} classes")

Tiers assigned for 52 classes


In [4]:
# ============================================================
# SECTION 4 — Validate all 5 models
# ============================================================
results_global = {}   # name -> {precision, recall, map50, map5095}
results_ap50    = {}   # name -> per-class ap50 array

for name, weights_path in MODEL_PATHS.items():
    if not weights_path.exists():
        print(f"SKIP {name}: weights not found")
        continue

    print(f"Validating {name}...")
    m = YOLO(str(weights_path))
    res = m.val(
        data=str(yaml_path), imgsz=640, batch=32, device=0,
        verbose=False, plots=True,
        project=str(WORK / "val_runs"), name=name.replace(" ", "_"),
    )

    results_global[name] = {
        "precision": res.box.mp,
        "recall":    res.box.mr,
        "map50":     res.box.map50,
        "map5095":   res.box.map,
    }
    results_ap50[name] = res.box.ap50

print("\nAll models validated.")

Validating Model B...
Ultralytics 8.4.91 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 122 layers, 2,384,976 parameters, 0 gradients, 5.2 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 5.3±2.5 ms, read: 25.6±8.1 MB/s, size: 162.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive/labels... 639 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 639/639 176.5it/s 3.6s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 3.7it/s 5.5s0.2s
                   all        639       1645      0.888      0.875      0.949      0.717
Speed: 0.9ms preprocess, 2.5ms inference, 0.0ms loss, 0.

In [5]:
# ============================================================
# SECTION 5 — Global comparison table
# ============================================================
print(f"{'Model':<16} {'Precision':>10} {'Recall':>10} {'mAP@0.5':>10} {'mAP@.5:.95':>12}")
print("-" * 62)
for name, g in results_global.items():
    print(f"{name:<16} {g['precision']:>10.4f} {g['recall']:>10.4f} "
          f"{g['map50']:>10.4f} {g['map5095']:>12.4f}")

Model             Precision     Recall    mAP@0.5   mAP@.5:.95
--------------------------------------------------------------
Model B              0.8882     0.8751     0.9487       0.7171
Model A              0.8876     0.8266     0.8922       0.6690
A_v2_widegap         0.8677     0.8039     0.8735       0.6529
A_v3_highcls         0.9121     0.8294     0.9219       0.6783
A_v4_combined        0.8896     0.8562     0.9583       0.7160


In [6]:
# ============================================================
# SECTION 6 — Tier-level comparison table
# ============================================================
print(f"{'Model':<16} {'Freq':>8} {'Med':>8} {'Rare':>8}")
print("-" * 44)
for name, ap in results_ap50.items():
    print(f"{name:<16} {tier_mean(ap,'Frequent'):>8.4f} "
          f"{tier_mean(ap,'Medium'):>8.4f} {tier_mean(ap,'Rare'):>8.4f}")

Model                Freq      Med     Rare
--------------------------------------------
Model B            0.9553   0.9420   0.9443
Model A            0.9549   0.9267   0.7365
A_v2_widegap       0.9521   0.9178   0.6773
A_v3_highcls       0.9670   0.9426   0.8150
A_v4_combined      0.9651   0.9435   0.9627


In [7]:
# ============================================================
# SECTION 7 — Wilcoxon: every config vs Model B
# ============================================================
if "Model B" not in results_ap50:
    raise RuntimeError("Model B results missing — cannot run Wilcoxon comparisons")

all_b = [results_ap50["Model B"][i] for i in range(n_classes)]

print(f"{'Config':<16} {'Global p':>10} {'Freq p':>10} {'Med p':>10} {'Rare p':>10}")
print("-" * 60)
for name, ap in results_ap50.items():
    if name == "Model B":
        continue
    all_v = [ap[i] for i in range(n_classes)]
    _, p_global = stats.wilcoxon(all_b, all_v)

    p_tiers = {}
    for tier in ["Frequent", "Medium", "Rare"]:
        ids = [i for i in range(n_classes) if TIER[i] == tier]
        b_vals = [results_ap50["Model B"][i] for i in ids]
        v_vals = [ap[i] for i in ids]
        _, p_t = stats.wilcoxon(b_vals, v_vals)
        p_tiers[tier] = p_t

    print(f"{name:<16} {p_global:>10.4f} {p_tiers['Frequent']:>10.4f} "
          f"{p_tiers['Medium']:>10.4f} {p_tiers['Rare']:>10.4f}")

Config             Global p     Freq p      Med p     Rare p
------------------------------------------------------------
Model A              0.1501     0.9218     0.1361     0.1230
A_v2_widegap         0.0199     0.4061     0.2477     0.0327
A_v3_highcls         0.2507     0.0059     0.8753     0.2930
A_v4_combined        0.0265     0.0138     0.7989     0.0781


In [8]:
# ============================================================
# SECTION 8 — Identify winner + full per-class breakdown vs Model B
# ============================================================
best_name, best_gap = None, -np.inf
rare_b = tier_mean(results_ap50["Model B"], "Rare")
for name, ap in results_ap50.items():
    if name == "Model B":
        continue
    gap = tier_mean(ap, "Rare") - rare_b
    if gap > best_gap:
        best_gap = gap
        best_name = name

print(f"Best config by Rare-tier delta: {best_name} ({best_gap:+.4f})\n")

ap_winner = results_ap50[best_name]
print(f"{'ID':<5} {'Class':<30} {'Model B':>8} {best_name:>14} {'Delta':>8}  Tier")
print("-" * 78)
for i in range(n_classes):
    ap_b, ap_v = results_ap50["Model B"][i], ap_winner[i]
    delta = ap_v - ap_b
    flag = "UP" if delta > 0.005 else ("DOWN" if delta < -0.005 else "--")
    print(f"{i:<5} {class_names[i]:<30} {ap_b:>8.4f} {ap_v:>14.4f} {delta:>+8.4f} {flag} {TIER[i]}")

Best config by Rare-tier delta: A_v4_combined (+0.0184)

ID    Class                           Model B  A_v4_combined    Delta  Tier
------------------------------------------------------------------------------
0     Pedestrian Crossing              0.9504         0.9739  +0.0235 UP Frequent
1     Equal-level Intersection         0.8950         0.8979  +0.0029 -- Rare
2     No Entry                         0.9774         0.9585  -0.0189 DOWN Frequent
3     Right Turn Only                  0.9479         0.9554  +0.0076 UP Frequent
4     Intersection                     0.9950         0.9617  -0.0333 DOWN Medium
5     Intersection with a non-priority road   0.9678         0.9807  +0.0130 UP Frequent
6     Danger zone on the left          0.9460         0.9197  -0.0263 DOWN Frequent
7     No Left Turn                     0.9727         0.9664  -0.0063 DOWN Frequent
8     Bus Stop                         0.9467         0.9743  +0.0276 UP Frequent
9     Roundabout                       0.

In [9]:
# ============================================================
# SECTION 9 — Save everything for the paper
# ============================================================
summary_rows = []
for name, g in results_global.items():
    row = {
        "model": name,
        "precision": g["precision"], "recall": g["recall"],
        "map50": g["map50"], "map5095": g["map5095"],
        "freq_tier": tier_mean(results_ap50[name], "Frequent"),
        "med_tier":  tier_mean(results_ap50[name], "Medium"),
        "rare_tier": tier_mean(results_ap50[name], "Rare"),
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(WORK / "all_5_models_summary.csv", index=False)

per_class_df = pd.DataFrame({
    "class_id": range(n_classes),
    "class_name": class_names,
    "tier": [TIER[i] for i in range(n_classes)],
    **{name: ap for name, ap in results_ap50.items()}
})
per_class_df.to_csv(WORK / "all_5_models_per_class.csv", index=False)

print("Saved:")
print(f"  - all_5_models_summary.csv")
print(f"  - all_5_models_per_class.csv")
print(f"\nBest config: {best_name}")

Saved:
  - all_5_models_summary.csv
  - all_5_models_per_class.csv

Best config: A_v4_combined


In [10]:
# ============================================================
# SECTION 10 — Deep Dive: Why does A_v4_combined win/lose vs Model B?
# ============================================================
WINNER = "A_v4_combined"
ap_b = results_ap50["Model B"]
ap_w = results_ap50[WINNER]

# ── Per-class deltas, sorted ─────────────────────────────────
deltas = []
for i in range(n_classes):
    deltas.append({
        "class_id": i,
        "class_name": class_names[i],
        "tier": TIER[i],
        "instances": counts.get(i, 0),
        "model_B": ap_b[i],
        WINNER: ap_w[i],
        "delta": ap_w[i] - ap_b[i],
    })

delta_df = pd.DataFrame(deltas).sort_values("delta", ascending=False)

print("── TOP 10 GAINERS (A_v4 beats Model B) ─────────────────")
print(delta_df.head(10)[["class_name", "tier", "instances", "model_B", WINNER, "delta"]].to_string(index=False))

print("\n── TOP 10 LOSERS (A_v4 worse than Model B) ─────────────")
print(delta_df.tail(10)[["class_name", "tier", "instances", "model_B", WINNER, "delta"]].to_string(index=False))

# ── Aggregate: does the direction of change correlate with instance count? ──
print("\n── Correlation: delta vs instance count ─────────────────")
corr = delta_df["delta"].corr(delta_df["instances"])
print(f"  Pearson correlation (delta vs instance count): {corr:.4f}")
print("  (negative = rarer classes tend to improve MORE under A_v4)")

# ── Aggregate: does direction correlate with QCVN41 category? ──
CATEGORY_MAPPING = {
    2:0, 7:0, 10:0, 14:0, 16:0, 17:0, 18:0, 19:0, 20:0,
    23:0, 24:0, 29:0, 32:0, 34:0, 35:0, 36:0, 38:0, 39:0,
    40:0, 41:0, 46:0, 47:0, 49:0,
    0:1, 1:1, 4:1, 5:1, 6:1, 13:1, 15:1, 21:1, 22:1,
    26:1, 27:1, 33:1, 37:1, 43:1, 44:1, 48:1, 50:1, 51:1,
    3:2, 9:2, 12:2, 42:2,
    8:3, 11:3, 31:3, 45:3,
    25:4, 28:4, 30:4,
}
CATEGORY_NAMES = {0:"Prohibitory", 1:"Warning", 2:"Mandatory", 3:"Information", 4:"Supplementary"}
delta_df["category"] = delta_df["class_id"].map(CATEGORY_MAPPING).map(CATEGORY_NAMES)

print("\n── Mean delta by QCVN 41 category ───────────────────────")
print(delta_df.groupby("category")["delta"].agg(["mean", "count"]).sort_values("mean", ascending=False).to_string())

# ── How many category-siblings does each big mover have? ────
sibling_counts = {}
for cat_id in CATEGORY_NAMES:
    sibling_counts[cat_id] = sum(1 for v in CATEGORY_MAPPING.values() if v == cat_id)
delta_df["category_size"] = delta_df["class_id"].map(CATEGORY_MAPPING).map(sibling_counts)

print("\n── Do big gainers have MORE or FEWER category siblings? ──")
print(f"  Top 10 gainers - avg category size: {delta_df.head(10)['category_size'].mean():.1f}")
print(f"  Top 10 losers  - avg category size: {delta_df.tail(10)['category_size'].mean():.1f}")
print(f"  All classes    - avg category size: {delta_df['category_size'].mean():.1f}")

delta_df.to_csv(WORK / "deep_dive_deltas.csv", index=False)
print(f"\nSaved -> deep_dive_deltas.csv")

── TOP 10 GAINERS (A_v4 beats Model B) ─────────────────
                   class_name     tier  instances  model_B  A_v4_combined    delta
                    Left Turn     Rare         21 0.887857       0.995000 0.107143
  Double curve first to right     Rare         19 0.920000       0.995000 0.075000
        Trucks and Containers     Rare         27 0.759841       0.822833 0.062992
                  Trucks Only Frequent        138 0.925142       0.986579 0.061437
Road with Surveillance Camera   Medium         38 0.823346       0.883523 0.060177
            No Trucks Allowed   Medium         85 0.890703       0.949125 0.058422
       No Motorcycles Allowed   Medium         45 0.955000       0.995000 0.040000
         Speed limit (50km/h) Frequent        189 0.932120       0.967911 0.035792
                     Bus Stop Frequent        213 0.946712       0.974283 0.027571
 No Passenger Cars and Trucks Frequent        214 0.956411       0.980956 0.024545

── TOP 10 LOSERS (A_v4 worse 